# Historical simulation versus the four-stock SDE-GAN

Compare the three validation-selected GP-10 increments fits with fixed
historical simulation for 20-session, daily-rebalanced portfolio risk. The
validation review also checks marginal returns, cross-stock dependence, and
serial dependence. The notebook samples the saved models without retraining.

Run all cells with the repository's Python environment. The validation scores
must reproduce the saved 10,000-path results before opening the reserved split.
Enable `EVALUATE_TEST` to score the frozen protocol and write the test table.

In [ ]:
from pathlib import Path
from dataclasses import replace
from itertools import combinations
import json
import os
import sys

os.environ.setdefault("JAX_PLATFORMS", "cpu")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "benchmarks/historical.py").is_file())
experiment_dir = root / "experiments/4stocks"
data_dir = experiment_dir / "data"
runs_dir = experiment_dir / "results/sde_gan"
for path in [root, experiment_dir / "model"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from benchmarks.historical import (
    HistoricalSimulation, HistoricalSimulationConfig, _empirical_var, _empirical_es,
)
from sde_gan import SDEGAN

ablation_dir = experiment_dir / "results/increment_ablations"
CHECKPOINT_STEPS = (1500, 1000, 800)  # Lowest validation MMD² for seeds 0, 1, 2.
PLOT_SEED = 0
N_PATHS = 10_000
SAMPLE_SEED = 100
CONFIDENCE_LEVELS = (0.95, 0.99)
TEST_BLOCK_SESSIONS = 40
TEST_BOOTSTRAP_REPLICATES = 5_000
TEST_BOOTSTRAP_SEED = 20261119
EVALUATE_TEST = False
validation_reproduced = False

preprocessing = json.loads((data_dir / "preprocessing.json").read_text())
STOCKS = preprocessing["stock_order"]
HORIZON = preprocessing["window_length"]
WEIGHTS = np.full(len(STOCKS), 1 / len(STOCKS))
assert HORIZON == 20 and len(STOCKS) == 4 and preprocessing["transform"] == "log1p"
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)
print(f"Python: {sys.executable}\nHorizon: {HORIZON} sessions; weights: {WEIGHTS}")

## Comparison contract

- All inputs are **simple daily returns in fractions**, in the saved stock order.
  Generated standardized log returns are inverted using their paired training scaler.
- Each portfolio starts at equal 25% weights and rebalances every session:
  $R = \prod_{t=1}^{20}(1 + \sum_i w_i r_{t,i}) - 1$, with loss $L=-R$.
  We do not compound weighted individual log returns.
- Historical simulation uses **all complete contiguous training blocks**, stride
  one, without deleting missing rows or bridging gaps. It is fitted only on
  training data and remains fixed during validation and test, matching the unconditional
  generator's fixed scenario distribution. No validation observations enter
  either forecast distribution during this notebook.
- VaR uses the first loss whose empirical CDF reaches the confidence level.
  ES averages exactly the worst $1-c$ probability mass, including a fractional
  boundary observation. Both methods reuse the implementations in
  `historical.py`; old CSVs using interpolated quantiles are not mixed in.
- Historical blocks and evaluation targets overlap. Daily summaries pool
  windows, repeating some dates. Scenario counts and tail masses are **not
  independent sample sizes**. More generated paths reduce Monte Carlo noise,
  not uncertainty about the fitted market distribution.

In [ ]:
splits = ("train", "validation")


def read_returns(split: str) -> pd.DataFrame:
    """Read one chronological return split."""
    return pd.read_csv(data_dir / f"{split}_returns.csv", parse_dates=["Date"])


returns = {split: read_returns(split) for split in splits}
window_dates = pd.read_csv(data_dir / "window_dates.csv",
                           parse_dates=["start_date", "end_date"])


def to_simple(paths: np.ndarray, metadata: dict) -> np.ndarray:
    """Invert the paired scaler into finite simple returns, preserving path axes."""
    if metadata != preprocessing:
        raise ValueError("Checkpoint and prepared-data preprocessing differ")
    if paths.ndim != 3 or paths.shape[1:] != (HORIZON, len(STOCKS)):
        raise ValueError("Unexpected sample shape")
    with np.errstate(over="raise", invalid="raise"):
        simple = np.expm1(paths.astype(np.float64) * np.asarray(metadata["training_std"])
                          + np.asarray(metadata["training_mean"]))
    if not np.isfinite(simple).all():
        raise ValueError("Nonfinite inverse-transformed samples")
    return simple


def observed_windows(split: str) -> np.ndarray:
    """Read retained original CSV blocks without removing or filling missing rows."""
    frame = returns[split].set_index("Date").loc[:, STOCKS]
    dates = window_dates.loc[window_dates.split.eq(split)].sort_values("window_index")
    paths = np.stack([frame.loc[r.start_date:r.end_date].to_numpy()
                      for r in dates.itertuples()])
    if paths.shape != (preprocessing["splits"][split]["retained"], HORIZON, len(STOCKS)):
        raise ValueError(f"Unexpected {split} window shape")
    assert np.isfinite(paths).all()
    with np.load(data_dir / "prepared_windows.npz", allow_pickle=False) as prepared:
        np.testing.assert_allclose(to_simple(prepared[f"{split}_windows"], preprocessing),
                                   paths, rtol=1e-6, atol=1e-7)
    return paths


observed = {split: observed_windows(split) for split in splits}
for previous, following in zip(splits, splits[1:]):
    assert returns[previous].Date.max() < returns[following].Date.min()
display(pd.DataFrame([{
    "split": split, "first_date": returns[split].Date.min(),
    "last_date": returns[split].Date.max(), "raw_sessions": len(returns[split]),
    "complete_windows": len(observed[split]),
    "excluded_windows": len(returns[split]) - HORIZON + 1 - len(observed[split]),
} for split in splits]).set_index("split"))

## Run the historical benchmark

`historical.py` is an API: it does not save results itself. Run its public
`fit`, `summarize`, and `backtest` methods below. The date and compounding checks
ensure that the benchmark scenarios exactly match the windows seen by the GAN.
The historical distribution is used in full, rather than introducing randomness
by resampling it to match the generator's path count.


In [ ]:
config = HistoricalSimulationConfig(
    assets=tuple(STOCKS), weights=dict(zip(STOCKS, WEIGHTS)),
    return_type="simple", horizon_sessions=HORIZON, rebalance="daily",
    confidence_levels=CONFIDENCE_LEVELS, estimation="fixed", min_scenarios=100,
)
historical = HistoricalSimulation(config).fit(returns["train"])
historical_summary = historical.summarize()


def terminal_returns(paths: np.ndarray) -> np.ndarray:
    """Compound the daily-rebalanced portfolio in original return units."""
    result = np.prod(1 + paths @ WEIGHTS, axis=1) - 1
    assert np.isfinite(result).all()
    return result


train_dates = window_dates.loc[window_dates.split.eq("train")].sort_values("window_index")
np.testing.assert_array_equal(historical_summary.scenarios.scenario_start,
                              train_dates.start_date)
np.testing.assert_array_equal(historical_summary.scenarios.scenario_end,
                              train_dates.end_date)
np.testing.assert_allclose(historical_summary.scenarios.portfolio_return,
                          terminal_returns(observed["train"]), rtol=0, atol=1e-14)
fixture = np.zeros((1, HORIZON, len(STOCKS)))
fixture[0, 0, 0], fixture[0, 1, 1] = 0.10, -0.10
np.testing.assert_allclose(terminal_returns(fixture), [1.025 * 0.975 - 1])
display(historical_summary.risk)
display(pd.Series({key: historical_summary.metadata[key] for key in
                   ["quantile_convention", "es_convention", "missing_block_policy"]}))


## Sample the selected checkpoints

Load the minimum-validation-MMD² checkpoint for each fit. Draw 10,000 paths
per fit with sampling seed 100 and solver step 0.25. A paired draw at step
0.125 checks numerical sensitivity. Tables keep all fits visible. Plots use
seed 0 for readability.

In [ ]:
generated, fine_generated, inventory = {}, {}, []
for seed, step in enumerate(CHECKPOINT_STEPS):
    name = f"increments / seed {seed}"
    run_dir = runs_dir / "increments" / f"seed_{seed}"
    metadata = json.loads((run_dir / "preprocessing.json").read_text())
    model = SDEGAN.load(run_dir)
    run = model.run_metadata
    settings = model.config
    assert (run["fit_seed"], run["selected_step"], run["selection_rule"]) == (
        seed, step, "lowest_validation_mmd2"
    ), name
    assert settings.generator_output == "increments" and settings.generator_dt == 0.25
    assert settings.gradient_penalty_weight == 10
    assert (settings.temporal_penalty_weight == settings.absolute_temporal_penalty_weight
            == settings.signature_loss_weight == 0)
    print(f"Sampling {name} at checkpoint {step}", flush=True)
    generated[name] = to_simple(model.sample(N_PATHS, SAMPLE_SEED), metadata)
    model.config = replace(settings, generator_dt=0.125)
    fine_generated[name] = to_simple(model.sample(N_PATHS, SAMPLE_SEED), metadata)
    inventory.append({
        "dataset": name, "seed": seed, "selected_step": step,
        "validation_mmd2": run["validation_score"], "paths": len(generated[name]),
        "checkpoint": str(run_dir.relative_to(root)),
    })

display(pd.DataFrame(inventory).set_index("dataset"))
datasets = {"historical simulation": observed["train"],
            "validation (observed)": observed["validation"], **generated}
representatives = [f"increments / seed {PLOT_SEED}"]
plot_names = ["historical simulation", "validation (observed)", *representatives]

## Marginal returns and exact zeros

Means, standard deviations and quantiles are reported in percent per session;
`zero_pct` is a percentage of observations. Pooling historical windows describes
the empirical distribution actually used for training, not equally weighted
unique market dates. A continuous generator does not explicitly model a point
mass at zero. Compare that discrepancy separately from tail or volatility fit.
Tables include all seeds; figures use the preselected seed 0 for readability.


In [ ]:
flat = {name: pd.DataFrame(paths.reshape(-1, len(STOCKS)), columns=STOCKS)
        for name, paths in datasets.items()}
marginals = pd.concat({name: pd.DataFrame({
    "mean_pct": frame.mean() * 100, "std_pct": frame.std(ddof=1) * 100,
    "q01_pct": frame.quantile(0.01) * 100, "q05_pct": frame.quantile(0.05) * 100,
    "q50_pct": frame.quantile(0.50) * 100, "q95_pct": frame.quantile(0.95) * 100,
    "q99_pct": frame.quantile(0.99) * 100, "zero_pct": frame.eq(0).mean() * 100,
}) for name, frame in flat.items()}, names=["dataset", "stock"])
display(marginals.round(4))
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for column, title, ax in zip(["std_pct", "zero_pct"],
                            ["Daily volatility (%)", "Exact zero returns (%)"], axes):
    marginals[column].unstack("stock").loc[plot_names].T.plot.bar(ax=ax)
    ax.set(title=title, xlabel="")
    ax.tick_params(axis="x", rotation=18)
    ax.legend(fontsize=7)
plt.show()
plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
for stock, ax in zip(STOCKS, axes.flat):
    for name in plot_names:
        values = np.sort(flat[name][stock].to_numpy()) * 100
        ax.plot(values, np.arange(1, len(values) + 1) / len(values), label=name)
    ax.set(title=stock, xlabel="Daily simple return (%)", ylabel="Empirical CDF")
    ax.legend(fontsize=7)
plt.show()
plt.close(fig)


## Cross-stock dependence and joint downside events

Pearson and average-rank Spearman correlations measure contemporaneous
dependence. Joint downside frequencies use each stock's **training-window 5%
quantile**, held fixed for every dataset. Report the number of pooled daily
observations alongside the event count. These are finite-threshold co-loss
frequencies, not estimates of asymptotic tail dependence or independent counts.


In [ ]:
pearson = {name: frame.corr() for name, frame in flat.items()}
spearman = {name: frame.corr(method="spearman") for name, frame in flat.items()}
pairs = list(combinations(range(len(STOCKS)), 2))
dependence = pd.DataFrame([
    {"dataset": name, "pair": f"{STOCKS[i]} / {STOCKS[j]}",
     "pearson": pearson[name].iloc[i, j], "spearman": spearman[name].iloc[i, j]}
    for name in datasets for i, j in pairs
]).set_index(["dataset", "pair"])
display(dependence.round(3))

thresholds = flat["historical simulation"].quantile(0.05).to_numpy()
display(pd.Series(thresholds * 100, index=STOCKS, name="Training downside threshold (%)"))
joint_downside = []
for name, frame in flat.items():
    events = frame.to_numpy() <= thresholds
    for group in [*pairs, tuple(range(len(STOCKS)))]:
        count = int(events[:, group].all(axis=1).sum())
        joint_downside.append({"dataset": name, "stocks": " / ".join(STOCKS[i] for i in group),
                               "count": count, "pooled_observations": len(frame),
                               "frequency_pct": 100 * count / len(frame)})
joint_downside = pd.DataFrame(joint_downside).set_index(["dataset", "stocks"])
display(joint_downside.round(3))

fig, axes = plt.subplots(1, len(plot_names), figsize=(18, 4), constrained_layout=True)
for name, ax in zip(plot_names, axes):
    image = ax.imshow(pearson[name], vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set(title=name.replace(" / ", "\n"), xticks=range(4), yticks=range(4),
           xticklabels=["KOG", "FRO", "WWIB", "TOM"],
           yticklabels=["KOG", "FRO", "WWIB", "TOM"])
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{pearson[name].iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(image, ax=axes, shrink=0.7, label="Pearson correlation")
plt.show()
plt.close(fig)


## Temporal dependence

Correlate aligned pairs at lags 1–5 **within each path**, pooling pairs without
ever joining the end of one path to the next. Signed-return correlations assess
return persistence; absolute-return correlations assess persistence of return
magnitudes. Check both: matching signed returns alone does not establish that
volatility clustering is reproduced. Constant inputs yield unavailable
correlations, not artificially perfect agreement.


In [ ]:
def autocorrelation(paths: np.ndarray, absolute: bool = False) -> pd.DataFrame:
    """Compute pooled within-path lag correlations, preserving path boundaries."""
    values = np.abs(paths) if absolute else paths
    return pd.DataFrame({lag: pd.DataFrame(
        values[:, :-lag].reshape(-1, len(STOCKS)), columns=STOCKS
    ).corrwith(pd.DataFrame(values[:, lag:].reshape(-1, len(STOCKS)), columns=STOCKS))
        for lag in range(1, 6)}).T.rename_axis("lag")


alternating = np.broadcast_to(np.tile([1., -1.], 10)[None, :, None], (3, HORIZON, 4))
np.testing.assert_allclose(autocorrelation(alternating).loc[1], -1)
signed_acf = {name: autocorrelation(paths) for name, paths in datasets.items()}
absolute_acf = {name: autocorrelation(paths, absolute=True) for name, paths in datasets.items()}
display(pd.concat({"signed lag 1": pd.DataFrame({n: t.loc[1] for n, t in signed_acf.items()}).T,
                   "absolute lag 1": pd.DataFrame({n: t.loc[1] for n, t in absolute_acf.items()}).T},
                  names=["metric", "dataset"]).round(3))
for title, tables in [("Signed returns", signed_acf), ("Absolute returns", absolute_acf)]:
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
    for stock, ax in zip(STOCKS, axes.flat):
        for name in plot_names:
            ax.plot(tables[name].index, tables[name][stock], marker=".", label=name)
        ax.axhline(0, color="black", linewidth=0.5)
        ax.set(title=stock, xlabel="Lag (sessions)", ylabel="Correlation", xticks=range(1, 6))
        ax.legend(fontsize=7)
    fig.suptitle(f"{title}: within-window autocorrelation")
    plt.show()
    plt.close(fig)


## Portfolio loss distributions, VaR and ES

All risk values below are percentage losses over **20 sessions**, not annualized
daily figures. Negative VaR is retained if the relevant quantile represents a
gain. The observed validation distribution is a descriptive reference, never
used to set a forecast threshold. `tail_mass` is the empirical probability mass
expressed in numbers of scenarios; fewer than 20 is flagged, matching the
benchmark. ES is particularly sensitive to a small number of extreme losses.


In [ ]:
def risk_rows(losses: np.ndarray, name: str) -> list[dict]:
    """Use the benchmark's exact empirical VaR and fractional-mass ES conventions."""
    ordered = np.sort(losses)
    return [{"dataset": name, "confidence": confidence, "scenarios": len(ordered),
             "mean_loss_pct": ordered.mean() * 100, "std_loss_pct": ordered.std(ddof=1) * 100,
             "var_pct": _empirical_var(ordered, confidence) * 100,
             "es_pct": _empirical_es(ordered, confidence) * 100,
             "tail_mass": len(ordered) * (1 - confidence),
             "small_tail_sample": len(ordered) * (1 - confidence) < 20}
            for confidence in CONFIDENCE_LEVELS]


losses = {name: -terminal_returns(paths) for name, paths in datasets.items()}
risk = pd.DataFrame([row for name, values in losses.items()
                     for row in risk_rows(values, name)]).set_index(["dataset", "confidence"])
np.testing.assert_allclose(risk.loc["historical simulation", ["var_pct", "es_pct"]],
                          historical_summary.risk[["var", "es"]].to_numpy() * 100)
assert (risk.es_pct >= risk.var_pct - 1e-12).all()
# Independent fractional-tail fixture: worst 1.5 observations are 0.10 and half of 0.03.
np.testing.assert_allclose(_empirical_es(np.array([-0.02, 0.01, 0.03, 0.10]), 0.625),
                          (0.10 + 0.5 * 0.03) / 1.5)
display(risk.round(4))
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for name in plot_names:
    ordered = np.sort(losses[name]) * 100
    probability = np.arange(1, len(ordered) + 1) / len(ordered)
    for ax in axes:
        ax.plot(ordered, probability, label=name)
axes[0].set(title="20-session portfolio loss", xlabel="Loss (%)", ylabel="Empirical CDF")
axes[1].set(title="Upper loss tail", xlabel="Loss (%)", ylabel="Empirical CDF", ylim=(0.9, 1.001))
for ax in axes:
    ax.legend(fontsize=7)
plt.show()
plt.close(fig)


## Fixed-distribution validation backtest

For each validation start date, the benchmark forecasts a complete 20-session
target. Targets with missing returns remain in its output and are excluded from
**every method's** scores. All methods use exactly the same eligible dates,
realized losses, and strict breach rule `loss > VaR`. The historical forecast
remains fitted to training data; the generator uses its fixed sample quantile.

Coverage close to $1-c$ alone does not establish a good forecast. Also report
mean quantile (pinball) loss, $\rho_c(L-q)=(c-\mathbf{1}_{L<q})(L-q)$, where
smaller is better for predicting that loss quantile. Values are scaled by 100
for readability. No independence tests, binomial confidence intervals, or
significance claims are made for overlapping targets. The observed average loss
beyond VaR is descriptive and is not by itself a formal ES backtest.


In [ ]:
validation_reproduced = False
validation_backtest = historical.backtest(returns["validation"])
forecasts = validation_backtest.forecasts
eligible = forecasts.loc[
    forecasts.confidence.eq(CONFIDENCE_LEVELS[0])
    & forecasts.estimate_status.eq("ok") & forecasts.realization_status.eq("ok")
].copy()
validation_dates = window_dates.loc[window_dates.split.eq("validation")].sort_values("window_index")
np.testing.assert_array_equal(eligible.target_start, validation_dates.start_date)
np.testing.assert_array_equal(eligible.target_end, validation_dates.end_date)
np.testing.assert_allclose(eligible.realized_loss, losses["validation (observed)"], rtol=0, atol=1e-14)
assert (forecasts.history_end < forecasts.target_start).all()


def quantile_losses(realized: np.ndarray, threshold: float, confidence: float) -> np.ndarray:
    """Return the scaled pinball loss for each matched target."""
    residual = realized - threshold
    return 100 * (confidence - (residual < 0)) * residual


def score_forecasts(realized: np.ndarray, method_names: list[str]) -> pd.DataFrame:
    """Compare fixed thresholds on the same complete realized portfolio targets."""
    rows = []
    for name in method_names:
        for confidence in CONFIDENCE_LEVELS:
            estimate = risk.loc[(name, confidence)]
            threshold = estimate.var_pct / 100
            breaches = realized > threshold
            rows.append({
                "dataset": name, "confidence": confidence, "targets": len(realized),
                "var_pct": estimate.var_pct, "es_pct": estimate.es_pct,
                "breaches": int(breaches.sum()), "breach_rate_pct": breaches.mean() * 100,
                "nominal_rate_pct": (1 - confidence) * 100,
                "quantile_loss_pct": quantile_losses(realized, threshold, confidence).mean(),
                "mean_loss_beyond_var_pct": realized[breaches].mean() * 100 if breaches.any() else np.nan,
            })
    return pd.DataFrame(rows).set_index(["dataset", "confidence"])


method_names = ["historical simulation", *generated]
validation_scores = score_forecasts(eligible.realized_loss.to_numpy(), method_names)
np.testing.assert_array_equal(validation_scores.loc["historical simulation", "breaches"],
                              validation_backtest.coverage.n_breaches)
saved_gan = pd.read_csv(ablation_dir / "validation_risk_scores_10000_paths.csv")
saved_gan = saved_gan.loc[saved_gan.experiment.eq("increments")].copy()
assert saved_gan.n_paths.eq(N_PATHS).all() and saved_gan.sample_seed.eq(SAMPLE_SEED).all()
saved_gan["dataset"] = "increments / seed " + saved_gan.seed.astype(str)
saved_historical = pd.read_csv(ablation_dir / "validation_risk_scores.csv")
saved_historical = saved_historical.loc[saved_historical.experiment.eq("historical")].assign(
    dataset="historical simulation"
)
saved_validation = pd.concat([saved_historical, saved_gan]).set_index(["dataset", "confidence"])
pd.testing.assert_index_equal(validation_scores.index, saved_validation.index)
np.testing.assert_array_equal(validation_scores.breaches, saved_validation.breaches)
columns = ["var_pct", "es_pct", "breach_rate_pct", "quantile_loss_pct"]
np.testing.assert_allclose(validation_scores[columns], saved_validation[columns],
                           rtol=1e-6, atol=1e-7)
validation_reproduced = True
print("Reproduced the saved validation scores for historical simulation and all three fits.")

display(validation_backtest.coverage)
display(validation_scores.round(4))

plot_targets = forecasts.loc[forecasts.confidence.eq(CONFIDENCE_LEVELS[0])]
fig, axes = plt.subplots(1, len(CONFIDENCE_LEVELS), figsize=(15, 5), constrained_layout=True)
for confidence, ax in zip(CONFIDENCE_LEVELS, np.atleast_1d(axes)):
    ax.plot(plot_targets.target_start, plot_targets.realized_loss * 100, color="0.5", linewidth=1,
            label="realized validation loss")
    for index, name in enumerate(["historical simulation", *representatives]):
        ax.axhline(risk.loc[(name, confidence), "var_pct"], color=f"C{index}",
                   label=f"{name}: VaR")
    ax.set(title=f"{confidence:.0%} VaR over the same validation targets",
           xlabel="Target start date", ylabel="20-session loss (%)")
    ax.legend(fontsize=7)
plt.show()
plt.close(fig)

## Solver sensitivity and variation across fitted seeds

Compare VaR and ES from the paired samples at steps 0.25 and 0.125. The main
comparison uses 0.25. This check measures the effect of one refinement on the
risk estimates for these fits.

Report each fitted seed before summarizing its range. The fits share a single
market history, so their range describes training variability.

In [ ]:
solver_risk_rows = []
for name in generated:
    fine_table = pd.DataFrame(risk_rows(-terminal_returns(fine_generated[name]), name))
    for fine in fine_table.itertuples():
        coarse = risk.loc[(name, fine.confidence)]
        solver_risk_rows.append({
            "dataset": name, "confidence": fine.confidence,
            "var_fine_minus_coarse_pp": fine.var_pct - coarse.var_pct,
            "es_fine_minus_coarse_pp": fine.es_pct - coarse.es_pct,
        })
display(pd.DataFrame(solver_risk_rows).set_index(["dataset", "confidence"]).round(4))

seed_scores = validation_scores.loc[list(generated)].reset_index()
seed_ranges = seed_scores.groupby("confidence").agg(
    var_min_pct=("var_pct", "min"), var_max_pct=("var_pct", "max"),
    es_min_pct=("es_pct", "min"), es_max_pct=("es_pct", "max"),
    breach_min_pct=("breach_rate_pct", "min"), breach_max_pct=("breach_rate_pct", "max"),
    quantile_loss_min_pct=("quantile_loss_pct", "min"),
    quantile_loss_max_pct=("quantile_loss_pct", "max"),
)
display(seed_ranges.round(4))

## What do these results support?

The following review is computed from the tables above, so it changes when
inputs or saved runs change. Lower quantile loss supports a better quantile
forecast **on these validation targets**. It does not establish calibrated ES,
better performance on the reserved test set, or superiority across market regimes.
Inspect marginal volatility, signed and absolute dependence, and numerical
sensitivity alongside portfolio risk; a large VaR can simply reflect excessive
generated return persistence.


In [ ]:
review = [f"**Validation review:** {len(eligible)} complete, overlapping {HORIZON}-session targets."]
for confidence in CONFIDENCE_LEVELS:
    baseline = validation_scores.loc[("historical simulation", confidence)]
    current = seed_scores.loc[seed_scores.confidence.eq(confidence)]
    better = int((current.quantile_loss_pct < baseline.quantile_loss_pct).sum())
    review.append(
        f"- At {confidence:.0%}, historical simulation has VaR **{baseline.var_pct:.2f}%**, "
        f"ES **{baseline.es_pct:.2f}%**, and **{int(baseline.breaches)}/{int(baseline.targets)}** "
        f"breaches ({baseline.breach_rate_pct:.2f}%; nominal {(1-confidence)*100:.2f}%). "
        f"**{better}/{len(current)}** available model runs have lower validation quantile loss."
    )
review.append("\n**Signed lag-1 persistence (range across the four stocks):**")
for name in plot_names:
    lag_one = signed_acf[name].loc[1]
    review.append(f"- {name}: {lag_one.min():.3f} to {lag_one.max():.3f}.")
observed_upper = signed_acf["validation (observed)"].loc[1].max()
persistent = [name for name in representatives if signed_acf[name].loc[1].min() > observed_upper]
if persistent:
    review.append("\nAll four stocks have higher lag-1 correlation than any observed validation "
                  "stock in these representative runs: " + "; ".join(persistent) + ". "
                  "Generated signed-return persistence remains a substantial mismatch. "
                  "Persistence can affect compounded portfolio tails; these diagnostics "
                  "do not isolate its contribution from volatility and cross-stock dependence.")
review.append(
    f"\nThe {max(CONFIDENCE_LEVELS):.0%} historical tail contains "
    f"{risk.loc[('historical simulation', max(CONFIDENCE_LEVELS)), 'tail_mass']:.2f} "
    "scenario-equivalents, and the scenarios overlap. Treat rare-loss and ES conclusions as provisional. "
    "Validation has influenced checkpoint selection; freeze choices before final test evaluation."
)
display(Markdown("\n".join(review)))

## Reserved-test protocol

Run this evaluation after reproducing validation. The forecasts stay fixed
and reuse the same distributions used in validation, which selected the generator checkpoints. Score
every GP-10 fit on the same dated targets as historical simulation.

The primary comparison uses 95% VaR pinball loss and the breach rate relative
to 5%. Lower pinball loss favors the model. Report the difference against
historical per fit and averaged across the three fits' scores. Per-fit
results show training variability. The average summarizes the three fits' scores.

For the primary comparison, use percentile intervals from 5,000 paired
circular moving-block bootstrap replicates with 40 consecutive target starts
per block. These blocks span twice the forecast horizon. Resample the same
starts for all methods and hold fitted models fixed. Intervals describe test
period uncertainty conditional on those fits. Treat 99% coverage as
descriptive given the limited tail observations. Report ES as an estimate
from the scenario distribution.

In [ ]:
if EVALUATE_TEST:
    if not validation_reproduced:
        raise RuntimeError("Reproduce the saved validation scores before evaluating test returns.")
    returns["test"] = read_returns("test")
    assert returns["validation"].Date.max() < returns["test"].Date.min()
    test_backtest = historical.backtest(returns["test"])
    test_targets = test_backtest.forecasts.loc[
        test_backtest.forecasts.confidence.eq(CONFIDENCE_LEVELS[0])
        & test_backtest.forecasts.estimate_status.eq("ok")
        & test_backtest.forecasts.realization_status.eq("ok")
    ]
    test_dates = window_dates.loc[window_dates.split.eq("test")].sort_values("window_index")
    np.testing.assert_array_equal(test_targets.target_start, test_dates.start_date)
    np.testing.assert_array_equal(test_targets.target_end, test_dates.end_date)
    test_sessions = returns["test"].Date
    all_target_starts = test_sessions.iloc[:len(test_sessions) - HORIZON + 1]
    np.testing.assert_array_equal(test_targets.target_start, all_target_starts)
    realized = test_targets.realized_loss.to_numpy()
    np.testing.assert_allclose(realized, -terminal_returns(observed_windows("test")),
                               rtol=0, atol=1e-14)
    assert (test_targets.history_end < test_targets.target_start).all()

    test_scores = score_forecasts(realized, method_names)
    historical_name = "historical simulation"
    np.testing.assert_array_equal(test_scores.loc[historical_name, "breaches"],
                                  test_backtest.coverage.n_breaches)
    mean_name = "increments / mean of seed scores"
    mean_columns = ["breach_rate_pct", "quantile_loss_pct"]
    across_seeds = test_scores.loc[list(generated), mean_columns].groupby("confidence").mean()
    test_report = pd.concat([test_scores, pd.concat({mean_name: across_seeds}, names=["dataset"])])
    test_report["targets"] = len(realized)
    test_report["nominal_rate_pct"] = (1 - test_report.index.get_level_values("confidence")) * 100
    test_report["loss_difference_vs_historical_pp"] = test_report.quantile_loss_pct.sub(
        test_scores.loc[historical_name, "quantile_loss_pct"], level="confidence"
    )

    n_targets = len(realized)
    rng = np.random.default_rng(TEST_BOOTSTRAP_SEED)
    block_count = (n_targets + TEST_BLOCK_SESSIONS - 1) // TEST_BLOCK_SESSIONS
    starts = rng.integers(n_targets, size=(TEST_BOOTSTRAP_REPLICATES, block_count))
    blocks = (starts[..., None] + np.arange(TEST_BLOCK_SESSIONS)) % n_targets
    indices = blocks.reshape(TEST_BOOTSTRAP_REPLICATES, -1)[:, :n_targets]

    primary = CONFIDENCE_LEVELS[0]
    thresholds = test_scores.xs(primary, level="confidence").var_pct / 100
    per_target_loss = pd.DataFrame({
        name: quantile_losses(realized, threshold, primary)
        for name, threshold in thresholds.items()
    })
    loss_difference = per_target_loss.sub(per_target_loss[historical_name], axis=0)
    breach_indicators = pd.DataFrame(100 * (realized[:, None] > thresholds.to_numpy()),
                                     columns=thresholds.index)
    for values, prefix in ((loss_difference, "loss_difference"), (breach_indicators, "breach_rate")):
        values[mean_name] = values[list(generated)].mean(axis=1)
        for name in values:
            draws = values[name].to_numpy()[indices].mean(axis=1)
            low, high = np.quantile(draws, (0.025, 0.975))
            test_report.loc[(name, primary), f"{prefix}_ci95_low"] = low
            test_report.loc[(name, primary), f"{prefix}_ci95_high"] = high

    output = ablation_dir / "reserved_test_gp10_vs_historical.csv"
    test_report.to_csv(output)
    print(f"Scored {n_targets} test targets from {test_targets.target_start.min().date()} "
          f"through {test_targets.target_end.max().date()}.")
    display(test_report.round(4))
else:
    print("Validation review complete. Set EVALUATE_TEST = True to run the reserved test.")